In [1]:
!apt-get update -qq
!apt-get install -y stockfish wget
!pip install -q python-chess zstandard tensorflow scikit-learn tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.2-2ubuntu1.1).
Suggested packages:
  polyglot xboard | scid
The following NEW packages will be installed:
  stockfish
0 upgraded, 1 newly installed, 0 to remove and 62 not upgraded.
Need to get 24.8 MB of archives.
After this operation, 47.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 stockfish amd64 14.1-1 [24.8 MB]
Fetched 24.8 MB in 0s (59.3 MB/s)
Selecting previously unselected package stockfish.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../stockfish_14.1-1_amd64.deb ...
Unpacking stockfish (14.1-1) ...
Setting up stockfish (14.1-1) ...
Processing trigg

In [2]:
import os
import io
import json
import math
import random
import zstandard as zstd
import numpy as np
import chess
import chess.pgn
import chess.engine
import tensorflow as tf

from tensorflow import keras
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from google.colab import files

STOCKFISH_PATH = "/usr/games/stockfish"

if not os.path.exists(STOCKFISH_PATH):
    raise FileNotFoundError("Stockfish not found at /usr/games/stockfish")

print("TensorFlow:", tf.__version__)
print("Stockfish:", STOCKFISH_PATH)

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

TensorFlow: 2.20.0
Stockfish: /usr/games/stockfish


In [12]:
PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2015-01.pgn.zst"
PGN_ZST = "lichess_db_standard_rated_2015-01.pgn.zst"
!wget -nc -O {PGN_ZST} "{PGN_URL}"
!ls -lh {PGN_ZST}

--2026-06-28 05:44:21--  https://database.lichess.org/standard/lichess_db_standard_rated_2015-01.pgn.zst
Resolving database.lichess.org (database.lichess.org)... 141.95.66.62, 2001:41d0:700:5e3e::
Connecting to database.lichess.org (database.lichess.org)|141.95.66.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 285559460 (272M) [application/octet-stream]
Saving to: ‘lichess_db_standard_rated_2015-01.pgn.zst’

lichess_db_standard 100%[===================>] 272.33M  26.7MB/s    in 11s     

2026-06-28 05:44:34 (24.2 MB/s) - ‘lichess_db_standard_rated_2015-01.pgn.zst’ saved [285559460/285559460]

-rw-r--r-- 1 root root 273M Nov  4  2022 lichess_db_standard_rated_2015-01.pgn.zst


In [14]:
def collect_pos(
    path,
    target_positions=30000,
    positions_per_game=2,
    min_ply=12,
    max_ply=90,
    min_rating=1400,
    max_games=200000,
):
    fens = []
    seen = set()
    games_seen = 0
    games_used = 0
    with open(path, "rb") as compressed:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
            while len(fens) < target_positions and games_seen < max_games:
                game = chess.pgn.read_game(text_stream)
                if game is None:
                    break
                games_seen += 1
                if games_seen % 5000 == 0:
                    print(
                        f"Games seen: {games_seen}, games used: {games_used}, positions: {len(fens)}"
                    )
                result = game.headers.get("Result", "")
                if result not in ["1-0", "0-1", "1/2-1/2"]:
                    continue

                try:
                    white_elo = int(game.headers.get("WhiteElo", "0"))
                    black_elo = int(game.headers.get("BlackElo", "0"))
                except ValueError:
                    continue
                if min(white_elo, black_elo) < min_rating:
                    continue
                board = game.board()
                candidate_fens = []
                for ply, move in enumerate(game.mainline_moves(), start=1):
                    if move not in board.legal_moves:
                        break
                    board.push(move)
                    if board.is_game_over():
                        break
                    if min_ply <= ply <= max_ply:
                        candidate_fens.append(board.fen())
                if not candidate_fens:
                    continue
                sampled = random.sample(
                    candidate_fens,
                    k=min(positions_per_game, len(candidate_fens))
                )
                added_any = False
                for fen in sampled:
                    if fen not in seen:
                        seen.add(fen)
                        fens.append(fen)
                        added_any = True
                    if len(fens) >= target_positions:
                        break
                if added_any:
                    games_used += 1
    return fens

TARGET_POSITIONS = 250000
fens = collect_pos(
    PGN_ZST,
    target_positions=TARGET_POSITIONS,
    positions_per_game=4,
    min_ply=10,
    max_ply=100,
    min_rating=1600,
    max_games=1000000,
)
print("Collected:", len(fens))

Games seen: 5000, games used: 1851, positions: 7380
Games seen: 10000, games used: 3978, positions: 15835
Games seen: 15000, games used: 6176, positions: 24591
Games seen: 20000, games used: 8396, positions: 33436
Games seen: 25000, games used: 10511, positions: 41859
Games seen: 30000, games used: 12655, positions: 50390
Games seen: 35000, games used: 14693, positions: 58506
Games seen: 40000, games used: 16689, positions: 66441
Games seen: 45000, games used: 18532, positions: 73762
Games seen: 50000, games used: 20340, positions: 80942
Games seen: 55000, games used: 22311, positions: 88767
Games seen: 60000, games used: 24372, positions: 96954
Games seen: 65000, games used: 26424, positions: 105087
Games seen: 70000, games used: 28533, positions: 113455
Games seen: 75000, games used: 30545, positions: 121442
Games seen: 80000, games used: 32588, positions: 129530
Games seen: 85000, games used: 34580, positions: 137442
Games seen: 90000, games used: 36552, positions: 145256
Games seen

In [15]:
PIECE_TO_PLANE = {
    (chess.PAWN, chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK, chess.WHITE): 3,
    (chess.QUEEN, chess.WHITE): 4,
    (chess.KING, chess.WHITE): 5,

    (chess.PAWN, chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK, chess.BLACK): 9,
    (chess.QUEEN, chess.BLACK): 10,
    (chess.KING, chess.BLACK): 11,
}
INPUT_SIZE = 773
def encode_board(board):
    x = np.zeros(INPUT_SIZE, dtype=np.float32)
    for square, piece in board.piece_map().items():
        plane = PIECE_TO_PLANE[(piece.piece_type, piece.color)]
        x[plane * 64 + square] = 1.0
    x[768] = 1.0 if board.turn == chess.WHITE else 0.0
    x[769] = 1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0
    x[770] = 1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0
    x[771] = 1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0
    x[772] = 1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0
    return x

In [17]:
STOCKFISH_DEPTH = 8
def stockfish_label(board, engine, depth=8):
    info = engine.analyse(board, chess.engine.Limit(depth=depth))
    score = info["score"].white()
    if score.is_mate():
        mate = score.mate()
        cp = 10000 if mate > 0 else -10000
    else:
        cp = score.score(mate_score=10000)
    cp = max(-2000, min(2000, cp))
    return np.tanh(cp / 600.0)

X = np.zeros((len(fens), INPUT_SIZE), dtype=np.float32)
y = np.zeros((len(fens),), dtype=np.float32)
with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
    try:
        engine.configure({"Threads": 2, "Hash": 512})
    except Exception as e:
        print("err:", e)
    for i, fen in enumerate(tqdm(fens)):
        board = chess.Board(fen)
        X[i] = encode_board(board)
        y[i] = stockfish_label(board, engine, depth=STOCKFISH_DEPTH)

print("Label min/max/mean:", y.min(), y.max(), y.mean())
np.savez_compressed("chess_eval_dataset_v2.npz", X=X, y=y, fens=np.array(fens))

  0%|          | 0/250000 [00:00<?, ?it/s]

Label min/max/mean: -0.997458 0.997458 0.036494326


In [21]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    shuffle=True
)
BATCH_SIZE = 1024
EPOCHS = 100
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=min(len(X_train), 50000), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
regularizer = keras.regularizers.l2(1e-5)
model = keras.Sequential([
    keras.layers.Input(shape=(INPUT_SIZE,)),
    keras.layers.Dense(
        512,
        activation="relu",
        kernel_regularizer=regularizer
    ),
    keras.layers.Dropout(0.05),
    keras.layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizer
    ),
    keras.layers.Dropout(0.05),
    keras.layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizer
    ),
    keras.layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizer
    ),
    keras.layers.Dense(1, activation="tanh"),
])
optimizer = keras.optimizers.AdamW(
    learning_rate=0.001,
    weight_decay=1e-5
)
model.compile(
    optimizer=optimizer,
    loss=keras.losses.Huber(delta=0.25),
    metrics=[
        "mae",
        keras.metrics.MeanSquaredError(name="mse")
    ]
)
class LearningRateLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        print(f"learning_rate: {lr:.8f}")
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00003,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "best_chess_eval_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    LearningRateLogger()
]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)
val_loss, val_mae, val_mse = model.evaluate(val_ds, verbose=0)
print("Validation loss:", val_loss)
print("Validation MAE:", val_mae)
print("Validation MSE:", val_mse)

Epoch 1/100
207/208 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - loss: 0.0812 - mae: 0.3840 - mse: 0.2565
Epoch 1: val_loss improved from None to 0.05437, saving model to best_chess_eval_model.keras

Epoch 1: finished saving model to best_chess_eval_model.keras
learning_rate: 0.00100000
208/208 ━━━━━━━━━━━━━━━━━━━━ 24s 89ms/step - loss: 0.0700 - mae: 0.3398 - mse: 0.2167 - val_loss: 0.0544 - val_mae: 0.2763 - val_mse: 0.1547 - learning_rate: 0.0010
Epoch 2/100
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 0.0517 - mae: 0.2654 - mse: 0.1446
Epoch 2: val_loss improved from 0.05437 to 0.04836, saving model to best_chess_eval_model.keras

Epoch 2: finished saving model to best_chess_eval_model.keras
learning_rate: 0.00100000
208/208 ━━━━━━━━━━━━━━━━━━━━ 20s 94ms/step - loss: 0.0497 - mae: 0.2569 - mse: 0.1374 - val_loss: 0.0484 - val_mae: 0.2522 - val_mse: 0.1340 - learning_rate: 0.0010
Epoch 3/100
207/208 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - loss: 0.0439 - mae: 0.2326 - mse: 0.1164
Epoch 3: val

In [22]:
MODEL_VERSION = "v2-lichess-stockfish-depth8"
export = {
    "input_size": INPUT_SIZE,
    "model_version": MODEL_VERSION,
    "layers": []
}
for layer in model.layers:
    if isinstance(layer, keras.layers.Dense):
        weights, bias = layer.get_weights()

        export["layers"].append({
            "name": layer.name,
            "activation": layer.activation.__name__,
            "weights": weights.astype(float).tolist(),
            "bias": bias.astype(float).tolist()
        })
os.makedirs("simple_model", exist_ok=True)
with open("simple_model/eval_weights.json", "w") as f:
    json.dump(export, f)
bot_info = {
    "modelVersion": MODEL_VERSION,
    "estimatedElo": None,
    "ratingMethod": "Not estimated yet",
    "searchDepth": 2,
    "trainingPositions": int(len(fens)),
    "stockfishLabelDepth": STOCKFISH_DEPTH,
    "validationMAE": float(val_mae),
}
with open("simple_model/bot_info.json", "w") as f:
    json.dump(bot_info, f, indent=2)
!ls -lh simple_model
files.download("simple_model/eval_weights.json")
files.download("simple_model/bot_info.json")

total 13M
-rw-r--r-- 1 root root 233 Jun 28 08:14 bot_info.json
-rw-r--r-- 1 root root 13M Jun 28 08:14 eval_weights.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>